# Proyecto de Codificación JPEG

**Asignatura:** Teoría de la Información y Procesado de Señal  
**Grado en Ciencia e Ingeniería de Datos (GCED). Universidad de A Coruña**  
**Modalidad:** Jupyter Notebook con asistencia de IA (incluye helper `jpeg_lib.py`)

---

## Objetivos de aprendizaje

En esta práctica crearemos un codificador JPEG, con cada una de sus partes: la conversión a *YCbCr*, la DCT, la cuantificación y la codificación entrópica. Es decir, implementaremos todas las fases que se pueden ver en la siguiente figura:

![JPEG encoding](../helpers/jpeg-encoding.png)

Al finalizar este proyecto serás capaz de:

1. Explicar la cadena básica de un codificador JPEG.
2. Implementar las operaciones de DCT (Discrete Cosine Transform), cuantificación e IDCT.
3. Analizar el efecto del parámetro de calidad y del subsampling sobre la reconstrucción.
4. Conocer los detalles de la codificación final: recorrido zig-zag, RLE y Huffman.
5. Evaluar el codificador con métricas de calidad, tasa y coste computacional.

---

## Reto central del proyecto

> ¿Qué partes de JPEG son reversibles, cuáles introducen pérdida y cómo cambia el compromiso calidad/tamaño/tiempo cuando modificamos la cuantificación y el subsampling?

## Metodología de trabajo con IA (recordatorio)

- La IA puede ayudarte a escribir código, pero tú debes validar y explicar cada resultado.
- En este proyecto no basta con obtener una imagen reconstruida: debes justificar qué se pierde, por qué se pierde y cómo medirlo.

---

## Identificación del estudiante

Completa los siguientes campos con tu información personal:

- **Apellidos:** _____

- **Nombre:** _____

- **Email UDC:** _____

In [ ]:
import time
from pathlib import Path
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.image as mpimg
from skimage import data
from skimage.color import rgb2ycbcr, ycbcr2rgb
from skimage.metrics import structural_similarity as ssim
import sys
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

try:
    # Add the helpers directory to the Python path
    helpers_path = Path('..') / 'helpers'
    if helpers_path.exists():
        sys.path.insert(0, str(helpers_path.resolve()))
    from jpeg_lib import huffman_decimal_to_binary
    print("✓ Helper jpeg_lib cargado")
except ImportError:
    print("⚠ No se encontró jpeg_lib.py")
    print("Asegúrate de que el archivo está en el directorio helpers al mismo nivel del directorio que contiene este cuaderno.")


np.random.seed(42)
print('✓ Entorno listo')


In [ ]:
def print_matrix(mat):
    for row in mat:
        parts = []
        for x in row:
            if isinstance(x, (np.floating, float)):
                parts.append(f"{float(x):.2f}")
            else:
                parts.append(str(int(x)))
        print("	".join(parts))


def plot_dct(dct_matrix: npt.NDArray[np.float64], title: str | None = None, vmax: float | None = None):
    values = np.log1p(np.abs(dct_matrix))
    plt.imshow(values, cmap='inferno', vmin=0, vmax=vmax)
    if title is not None:
        plt.title(title)
    #plt.colorbar(fraction=0.046, pad=0.04)
    plt.axis('off')


print('✓ Utilidades de visualización cargadas')

---

## Métricas de calidad y compresión

Para evaluar el rendimiento de un codificador JPEG necesitamos métricas que midan tanto la calidad de la imagen reconstruida como la eficiencia de la compresión. En esta sección introducimos las tres métricas principales que usaremos a lo largo del proyecto.

### Error cuadrático medio (MSE)

El MSE mide el promedio de los errores cuadráticos entre cada píxel de la imagen original y la reconstruida. Se define como:

$$
\text{MSE}(x, y) = \frac{1}{N \cdot M} \sum_{i=0}^{N-1} \sum_{j=0}^{M-1} [x(i,j) - y(i,j)]^2
$$

donde $x$ e $y$ son las imágenes original y reconstruida, $N$ y $M$ son sus dimensiones, y $x(i,j)$ representa el valor del píxel en la posición $(i,j)$.

El MSE tiene varias propiedades importantes. Es siempre no negativo, y vale cero solo cuando las imágenes son idénticas. Sin embargo, no está acotado superiormente y sus valores dependen del rango dinámico de la imagen. Un MSE de 100 en una imagen de 8 bits puede considerarse aceptable, mientras que el mismo valor en una imagen de 16 bits sería imperceptible. Por esta razón, el MSE por sí solo no nos dice mucho sobre la calidad perceptual.

### Índice de similitud estructural (SSIM)

El SSIM es una métrica diseñada para aproximar la percepción visual humana. A diferencia del MSE, que trata cada píxel de forma independiente, el SSIM considera la estructura local de la imagen comparando patrones de luminosidad, contraste y estructura.

La expresión completa del SSIM entre dos ventanas $x$ e $y$ es:

$$
\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}
$$

donde $\mu_x$ y $\mu_y$ son las medias locales, $\sigma_x^2$ y $\sigma_y^2$ son las varianzas, $\sigma_{xy}$ es la covarianza, y $C_1$ y $C_2$ son constantes de estabilidad que evitan división por cero.

El SSIM devuelve valores entre -1 y 1, donde 1 indica identidad estructural perfecta. Valores cercanos a 1 significan que la imagen reconstruida preserva la estructura visual de la original, lo cual se correlaciona mejor con la percepción humana que el MSE.

### Factor de compresión

El factor de compresión mide la reducción relativa del tamaño del archivo. Se define como:

$$
\text{Factor de compresión} = \frac{\text{tamaño original}}{\text{tamaño comprimido}}
$$

Un factor de compresión de 10 significa que el archivo comprimido es 10 veces más pequeño que el original. En JPEG, este factor depende principalmente de dos parámetros: el nivel de cuantificación (mayor cuantificación mayor compresión, menor calidad) y el tipo de subsampling cromático (4:4:4 preserva todo el color, 4:2:0 comprime más).

A continuación definimos las funciones que calcularemos estas métricas a lo largo del taller.


In [ ]:
def mse_metric(x: npt.NDArray[np.float64], y: npt.NDArray[np.float64]) -> float:
    return float(np.mean((x.astype(np.float64) - y.astype(np.float64)) ** 2))


def ssim_metric(x: npt.NDArray[np.float64], y: npt.NDArray[np.float64]) -> float:
    return float(ssim(x.astype(np.float64), y.astype(np.float64), data_range=255))


---

## 1. Transformación de color y análisis inicial de la imagen

El primer paso de un codificador JPEG consiste en convertir la imagen desde el espacio de color RGB al espacio YCbCr. Esta transformación separa la información de intensidad de la información de color: la componente $Y$ representa la luminancia, mientras que $Cb$ y $Cr$ representan dos componentes de crominancia.

La utilidad de esta separación es doble. Por un lado, nos permite estudiar mejor la estructura geométrica de la imagen, ya que los contornos y las variaciones de intensidad aparecen con claridad en $Y$. Por otro lado, hace posible reducir resolución en las componentes de color porque el sistema visual humano es mucho menos sensible a errores cromáticos que a errores de luminancia. Esa es la base del subsampling cromático usado en JPEG.

Es importante subrayar que esta conversión no introduce pérdida. La imagen sigue conteniendo la misma información, solo expresada en un sistema de coordenadas más conveniente para la compresión perceptual. La pérdida aparecerá más adelante, cuando cuantifiquemos coeficientes de la DCT.

En este taller usaremos varias imágenes de ejemplo incluidas en la biblioteca `skimage.data`, que proporciona imágenes de referencia habituales para experimentación y docencia en procesado de imagen. En este primer bloque vas a observar las componentes de la imagen completa. Después, en el bloque 1b, pasarás a seleccionar y estudiar un bloque de $8\times 8$ píxeles de luminancia, que será la unidad básica de trabajo del codificador.

### Bloque 1 · De RGB a YCbCr y visualización de componentes

**Reto conceptual:** identifica qué información visual está principalmente en $Y$ y cuál en $Cb/Cr$.

**Parámetros:**
- Imagen de trabajo inicial: `skimage.data.chelsea()` (imagen del gato)

**Hipótesis previa:**
- ¿Qué componente crees que sufrirá menos si reducimos su resolución espacial?
- ¿Qué tipo de detalles visuales esperas ver con más claridad en la luminancia?

**Tareas:**
1. Convierte la imagen RGB a YCbCr.
2. Visualiza RGB, Y, Cb y Cr por separado.
3. Compara visualmente qué información queda destacada en cada componente.

In [ ]:
# === IMPLEMENTACIÓN ===
image_rgb = data.chelsea()
image_ycbcr = rgb2ycbcr(image_rgb).astype(np.float32)

h, w = image_ycbcr.shape[:2]

h_even = h - (h % 2)
w_even = w - (w % 2)

image_ycbcr = image_ycbcr[:h_even, :w_even]


# TODO: separa las tres componentes y visualízalas


fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image_rgb)
axes[0].set_title('RGB')
axes[1].imshow(image_y, cmap='gray')
axes[1].set_title('Y')
axes[2].imshow(image_cb, cmap='viridis')
axes[2].set_title('Cb')
axes[3].imshow(image_cr, cmap='magma')
axes[3].set_title('Cr')
for ax in axes:
    ax.axis('off')
plt.tight_layout()


In [ ]:
# === VALIDACIÓN ===
assert image_rgb.ndim == 3 and image_rgb.shape[2] == 3
assert image_y.shape == image_cb.shape == image_cr.shape
print('✓ Conversión a YCbCr preparada')


### Explicación (OBLIGATORIA)

1. ¿Qué diferencias visuales observas entre $Y$, $Cb$ y $Cr$?

*Tu respuesta:*
```
[Escribe aquí]
```

2. ¿Por qué tiene sentido separar luminancia y crominancia antes de comprimir la imagen?

*Tu respuesta:*
```
[Escribe aquí]
```

---

## 2. Subsampling de crominancia

Una vez que disponemos de las componentes de crominancia separadas de la de luminancia, podemos reducir su tamaño y así rebajar el tamaño final de la imagen si afectar demasiado a la calidad final observada por el usuario. Esta redución se hace mediante el proceso de *downsampling*, que consiste en seleccionar solo algunos pixeles de la imagen. En la siguiente figura se pueden ver los modos admitidos:

![Ejemplo subsampling](../helpers/chroma-subsampling-ratio-examples.png)

Cada uno de los modos hace que las componentes de crominancia reduzcan su resolución horizontal, vertical o ambas, ya que por cada grupo de 8 pixeles se seleccionan solo 2 o 4. Por ejemplo, en el modo 4:2:0, por cada bloque de $2\times 4$ píxeles, se seleccionan dos píxeles de la primera fila y ninguno de la segunda, lo que reduce a la mitad la resolución horizontal y vertical. De igual forma, por ejemplo en el modo 4:1:1, se seleccionaría 1 solo pixel de la primera fila y otro de la segunda, lo que reduce a $\frac{1}{4}$ la resolución horizontal pero no cambia la vertical. El estándar JPEG no especifica el método exacto para obtener las muestras tras el subsampling, pero en la práctica es habitual utilizar un promedio local para evitar artefactos. Por ejemplo, en el modo 4:2:0, cada muestra de crominancia se obtiene promediando un bloque de $2\times 2$ píxeles originales.


### Bloque 2a · Subsampling 4:4:4, 4:2:2 y 4:2:0

**Idea clave:** el subsampling actua sobre la resolución espacial de las crominancias, eliminando información de color pero no de luminancia. El ojo humano es más sensible a errores en la luminancia que en la crominancia, por lo que podemos permitirnos reducir la resolución de las componentes de color sin afectar demasiado a la calidad percibida.

**Tareas:**
1. Implementa una función de subsampling para `Cb` y `Cr`.
2. Reconstruye una versión aproximada expandiendo de nuevo las crominancias.
3. Compara visualmente los resultados para 4:4:4, 4:2:2 y 4:2:0.
4. Usa el diagrama de referencia para interpretar correctamente cada modo.

In [ ]:
# === IMPLEMENTACIÓN ===

def subsample_channel(channel: npt.NDArray[np.float32], mode: str) -> npt.NDArray[np.float32]:
    if mode == '4:4:4':
        return channel.copy()

    if mode == '4:2:2':
        # Mitad de resolución horizontal
        return 0.5 * (channel[:, ::2] + channel[:, 1::2])
    
    if mode == '4:1:1':
        H, W = channel.shape
        W4 = (W // 4) * 4
        # Para que sea multiplo de 4
        channel_crop = channel[:, :W4]
        return 0.25 * (
            channel_crop[:, ::4] +
            channel_crop[:, 1::4] +
            channel_crop[:, 2::4] +
            channel_crop[:, 3::4]
        )

    if mode == '4:2:0':
        # Mitad de resolución horizontal y vertical
        return 0.25 * (
            channel[::2, ::2] +
            channel[1::2, ::2] +
            channel[::2, 1::2] +
            channel[1::2, 1::2]
        )

    raise ValueError('Modo no soportado')


def upsample_channel(
    channel_sub: npt.NDArray[np.float32],
    mode: str,
    target_shape: tuple[int, int]
) -> npt.NDArray[np.float32]:

    if mode == '4:4:4':
        return channel_sub.copy()
    
    if mode == '4:1:1':
        # Estirar solo horizontalmente (factor 4)
        return np.repeat(channel_sub, 4, axis=1)[:, :target_shape[1]]

    if mode == '4:2:2':
        # Estirar horizontalmente
        return np.repeat(channel_sub, 2, axis=1)[:, :target_shape[1]]

    if mode == '4:2:0':
        # Estirar horizontal y verticalmente
        return np.repeat(
            np.repeat(channel_sub, 2, axis=0),
            2,
            axis=1
        )[:target_shape[0], :target_shape[1]]

    raise ValueError('Modo no soportado')


modes = ['4:4:4', '4:2:2', '4:2:0']

H, W = image_cb.shape

fig, axes = plt.subplots(2, len(modes), figsize=(15, 8))

for col, mode in enumerate(modes):
    cb_sub = subsample_channel(image_cb.astype(np.float32), mode)
    cr_sub = subsample_channel(image_cr.astype(np.float32), mode)

    cb_up = upsample_channel(cb_sub, mode, image_cb.shape)
    cr_up = upsample_channel(cr_sub, mode, image_cr.shape)

    approx_ycbcr = np.stack([image_y, cb_up, cr_up], axis=2)
    approx_rgb = ycbcr2rgb(approx_ycbcr)

    # Canvas del tamaño original
    canvas = np.zeros((H, W), dtype=np.float32)

    h_sub, w_sub = cb_sub.shape
    canvas[:h_sub, :w_sub] = cb_sub

    # Fila superior: Cb con tamaño real dentro del canvas original
    axes[0, col].imshow(canvas, cmap='viridis', vmin=0, vmax=255)
    axes[0, col].set_title(
        f'Cb {mode}\n'
        f'resolución: {w_sub} x {h_sub}'
    )
    axes[0, col].axis('off')

    # Fila inferior: imagen reconstruida tras estirar Cb y Cr
    axes[1, col].imshow(np.clip(approx_rgb, 0, 1))
    axes[1, col].set_title(f'Reconstrucción {mode}')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

### Explicación (OBLIGATORIA)

1. ¿Por qué tiene sentido aplicar subsampling en crominancia pero no en luminancia?

*Tu respuesta:*
```
[Escribe aquí]
```

2. ¿Ves alguna diferencia entre las imágenes resultantes? ¿Qué significa eso respecto a la percepción visual de las componentes de color?

*Tu respuesta:*
```
[Escribe aquí]
```

### Bloque 2b · ¿Qué pasa si submuestreamos Y en vez de Cb/Cr?

**Tareas:**
1. Construye una versión 4:1:1 submuestreando solo `Cb` y `Cr` y dejando `Y` sin subsampling.
2. Construye otra versión 4:1:1 submuestreando solo `Y` y dejando `Cb` y `Cr` sin subsampling.
3. Reconstruye ambas aproximaciones y compáralas con la imagen RGB original.

In [ ]:
# === IMPLEMENTACIÓN ===
def to_rgb_from_ycbcr_stack(y_channel, cb_channel, cr_channel):
    stack = np.stack([y_channel, cb_channel, cr_channel], axis=2)
    return np.clip(ycbcr2rgb(stack), 0, 1)

mode = '4:2:0'
cb_sub = subsample_channel(image_cb.astype(np.float32), mode)
cr_sub = subsample_channel(image_cr.astype(np.float32), mode)
cb_up = upsample_channel(cb_sub, mode, image_cb.shape)
cr_up = upsample_channel(cr_sub, mode, image_cr.shape)
rgb_chroma_sub = to_rgb_from_ycbcr_stack(image_y, cb_up, cr_up)

y_sub = subsample_channel(image_y.astype(np.float32), mode)
y_up = upsample_channel(y_sub, mode, image_y.shape)
rgb_luma_sub = to_rgb_from_ycbcr_stack(y_up, image_cb, image_cr)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(image_rgb)
axes[0].set_title('Imagen original')
axes[1].imshow(rgb_chroma_sub)
axes[1].set_title('Subsampling 4:2:0 en Cb/Cr')
axes[2].imshow(rgb_luma_sub)
axes[2].set_title('Subsampling 4:2:0 en Y')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


### Explicación (OBLIGATORIA)

1. Viendo las imagenes resultantes, ¿cuál ves mas parecida a la original?  ¿Cual sería la explicación de esa diferencia? ¿En cuál de las dos imágenes se conserva más información?

*Tu respuesta:*
```
[Escribe aquí]
```

---

## 3. DCT e IDCT

La operación central de JPEG es la transformada del coseno discreto bidimensional o DCT 2D. Su objetivo es reescribir un bloque de $8\times 8$ píxeles como combinación de patrones espaciales de distinta frecuencia. Los coeficientes de baja frecuencia describen variaciones suaves de intensidad. los de alta frecuencia describen detalles finos, texturas y bordes abruptos.

Para un bloque centrado $s$, la DCT se define como
$$
S_{u,v} = \frac{1}{4} C_u C_v \sum_{x=0}^{7} \sum_{y=0}^{7} s_{x,y}
\cos\left(\frac{\pi (2x+1)u}{16}\right)
\cos\left(\frac{\pi (2y+1)v}{16}\right),
$$
donde $C_u$ y $C_v$ valen $1/\sqrt{2}$ cuando el índice correspondiente es 0, y 1 en otro caso.

Aunque esa expresión puede implementarse directamente, en la práctica es mucho más cómodo usar la formulación matricial
$$S = U \; s \; U^T,$$
y su inversa
$$s = U^T \; S \; U.$$

Donde $U$ es una matriz de $8\times 8$ que contiene los patrones de frecuencia:

$$
U_{k,n} =
\begin{cases}
\sqrt{\frac{1}{8}} & \text{si } k = 0 \\
\sqrt{\frac{2}{8}} \cos\left( \frac{(2n + 1)k\pi}{16} \right) & \text{si } k = 1,2,\dots,7
\end{cases}
$$

Una propiedad importante de esta etapa es que DCT e IDCT son reversibles si no cuantificamos. Por eso primero vamos a estudiar la transformada aislada y solo después introduciremos la pérdida.

In [ ]:
def compute_dct_matrix() -> npt.NDArray[np.float64]:
    U = np.zeros((8, 8), dtype=np.float64)
    U[0, :] = np.sqrt(2) / 4
    for u in range(1, 8):
        for x in range(8):
            U[u, x] = 0.5 * np.cos(((2 * x + 1) * u * np.pi) / 16)
    return U

U = compute_dct_matrix()
print('✓ Matriz de DCT preparada')

### Bloque 3a · Selección y análisis de un bloque

**Idea clave:** JPEG no transforma la imagen completa de una sola vez. En su lugar, la divide en bloques de $8\times 8$ píxeles y procesa cada bloque por separado. Observar un único bloque nos permite entender mejor qué representa la DCT y por qué los coeficientes bajos y altos tienen papeles distintos.

**Tareas:**
1. Selecciona un bloque de luminancia en las coordenadas que que quieras (sugerencia: elige una zona con detalles interesantes, como un borde o una textura).
2. Dibuja sobre la imagen original la posición del bloque seleccionado.
3. Muestra el bloque aislado e imprime su matriz de valores.

In [ ]:
# === IMPLEMENTACIÓN ===
#TODO: selecciona un bloque de 8x8 píxeles de la componente Y y visualízalo


fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image_rgb)
ax.add_patch(Rectangle((x, y), 8, 8, linewidth=2, edgecolor='red', facecolor='none'))
ax.set_title('Imagen original con el bloque seleccionado')
ax.axis('off')
plt.show()

plt.figure(figsize=(2.5, 2.5))
plt.imshow(block_y, cmap='gray', vmin=0, vmax=255)
plt.title(f'Bloque Y en (x={x}, y={y})')
plt.axis('off')
plt.show()

print('Matriz con los valores del bloque seleccionado:\n')
print_matrix(np.round(block_y).astype(int))


In [ ]:
# === VALIDACIÓN ===
assert block_y.shape == (8, 8)
assert x >= 0 and y >= 0
print('✓ Bloque seleccionado e inspeccionado')


### Bloque 3b · Centrado de los valores del bloque

Antes de aplicar la DCT, JPEG desplaza los niveles de gris desde el intervalo $[0,255]$ al intervalo aproximado $[-128,127]$. Este centrado hace que la transformada trabaje alrededor del valor medio y facilita la interpretación del coeficiente DC.

En una implementación real conviene tener cuidado con los tipos numéricos. Una matriz `uint8` no puede almacenar valores negativos, de modo que antes de restar 128 debemos convertir el bloque a coma flotante o a un tipo entero con signo.

**Tareas:**
1. Convierte el bloque a un tipo numérico con signo.
2. Resta 128 a cada píxel.
3. Imprime la matriz resultante y comprueba que los valores quedan ahora centrados alrededor de cero.

In [ ]:
# === IMPLEMENTACIÓN ===
#TODO: centra el bloque restando 128 a cada valor y visualízalo

print('Matriz del bloque centrada entre -128 y 127:\n')
print_matrix(np.round(block_centered_preview).astype(int))


In [ ]:
# === VALIDACIÓN ===
assert block_centered_preview.shape == (8, 8)
assert block_centered_preview.min() >= -128 - 1e-9
assert block_centered_preview.max() <= 127 + 1e-9
print('✓ Bloque centrado correctamente')


### Bloque 3c · Implementación de la DCT y la IDCT de un bloque

**Idea clave:** en este ejercicio vas a calcular la DCT de un bloque, observar sus coeficientes y comprobar que la IDCT recupera exactamente el bloque original mientras todavía no se ha aplicado cuantificación.

**Parámetros:**
- Bloque de entrada: `block_y`
- Matriz ortonormal: `U`

**Tareas:**
1. Centra el bloque restando 128.
2. Implementa `do_dct()` con producto matricial.
3. Implementa `do_idct()` y reconstruye el bloque.
4. Visualiza el bloque original, su DCT y la IDCT. Para visualizar la DCT se usa `log(1 + |DCT|)` porque los coeficientes tienen rangos de valores muy distintos. El valor absoluto muestra la magnitud sin tener en cuenta el signo, el logaritmo reduce la diferencia entre valores grandes y pequeños, y el `+1` evita calcular `log(0)`.
5. Imprime los valores numéricos para comparar la reconstrucción.

In [ ]:
# === IMPLEMENTACIÓN ===
block_centered = block_y.astype(np.float64) - 128.0

def do_dct(block: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    # TODO: implementa la DCT 2D usando la matriz U
    

def do_idct(block_dct: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    # TODO: implementa la IDCT 2D usando la matriz U
    

block_dct = do_dct(block_centered)
block_reconstructed = do_idct(block_dct) + 128.0

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(block_y, cmap='gray')
axes[0].set_title('Bloque original')
axes[1].imshow(np.log1p(np.abs(block_dct)), cmap='inferno')
axes[1].set_title('log(1 + |DCT|)')
axes[2].imshow(np.clip(block_reconstructed, 0, 255), cmap='gray')
axes[2].set_title('IDCT sin cuantificar')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Valores de la DCT del bloque:\n')
print_matrix(np.round(block_dct, 2))

print('Bloque recuperado con la IDCT:\n')
print_matrix(np.round(block_reconstructed).astype(int))


In [ ]:
# === VALIDACIÓN ===
assert block_centered.shape == (8, 8)
assert block_dct.shape == (8, 8)
assert np.max(np.abs(block_y - np.round(block_reconstructed))) <= 1
print('✓ DCT e IDCT verificadas en el bloque de ejemplo')

### Explicación (OBLIGATORIA)

1. ¿Por qué la DCT concentra la energía en las bajas frecuencias para muchos bloques naturales?

*Tu respuesta:*
```
[Escribe aquí]
```

2. ¿Qué representa el coeficiente DC y por qué suele ser tan importante?

*Tu respuesta:*
```
[Escribe aquí]
```

Una vez entendido el comportamiento de un bloque, el siguiente paso natural es aplicar la DCT a todos los bloques de la imagen. Al visualizar el valor absoluto de los coeficientes DCT de la imagen completa suelen apreciarse patrones: las regiones suaves concentran energía en la esquina superior izquierda de cada bloque, mientras que las zonas con textura o bordes reparten energía hacia frecuencias más altas.

Este ejercicio sirve para conectar el bloque aislado con el comportamiento global del codificador.

### Bloque 2b · DCT de una imagen completa

**Tareas:**
1. Recorta la imagen para que sus dimensiones sean múltiplos de 8.
2. Calcula la DCT de todos los bloques de la componente Y.
3. Visualiza la imagen original y `log(1 + |DCT|)` para interpretar qué zonas concentran más energía de baja frecuencia.

In [ ]:
def crop_to_multiple_of_8(image: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    h, w = image.shape[:2]
    return image[:h - (h % 8), :w - (w % 8)]


# Aplica la funcion fn a cada bloque de 8x8 de la imagen
def apply_blockwise(image: npt.NDArray[np.float64], fn):
    h, w = image.shape
    out = np.zeros_like(image, dtype=np.float64)
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            out[i:i+8, j:j+8] = fn(image[i:i+8, j:j+8])
    return out


def dct_of_image(image_centered: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    return apply_blockwise(image_centered, do_dct)


def idct_of_image(image_dct: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    return apply_blockwise(image_dct, do_idct)

print('✓ Funciones auxiliares de procesado por bloques preparadas')


In [ ]:
# === IMPLEMENTACIÓN ===
#TODO: aplica la DCT a toda la componente Y de la imagen y visualízala. 
# Recuerda que la DCT se aplica a bloques de 8x8, así que primero recorta
# la imagen para que sus dimensiones sean múltiplos de 8, 
# luego centra los valores restando 128, 
# y finalmente aplica la DCT a todos los bloques de la imagen


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(image_y_cropped, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Componente Y')
axes[1].imshow(np.log1p(np.abs(image_y_dct)), cmap='inferno')
axes[1].set_title('log(1 + |DCT|) de Y')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# === VALIDACIÓN ===
assert image_y_cropped.shape[0] % 8 == 0 and image_y_cropped.shape[1] % 8 == 0
assert image_y_dct.shape == image_y_cropped.shape
print('✓ DCT de imagen completa calculada')


### Explicación (OBLIGATORIA)

1. ¿Qué diferencias observas entre zonas suaves y zonas con bordes de la imagen cuando miras su DCT?

*Tu respuesta:*
```
[Escribe aquí]
```

2. ¿Por qué en la DCT se ven colores mas vivos en las zonas mas enfocadas de la imagen y menos intensos en las zonas borrosas?

*Tu respuesta:*
```
[Escribe aquí]
```

---

## 3. Cuantificación y pérdida de información

La cuantificación es la etapa con pérdida del codificador JPEG. Después de calcular la DCT, cada coeficiente se divide por un valor de una matriz de cuantificación y el resultado se redondea. Los coeficientes pequeños, especialmente en altas frecuencias, pasan con facilidad a cero.

Si denotamos por $S$ la matriz de coeficientes DCT y por $Q$ la matriz de cuantificación, la salida cuantificada se escribe como
$$K = \left\lfloor \frac{S}{Q} \right\rceil.$$

JPEG utiliza dos matrices de cuantificación distintas: una para luminancia ($Y$) y otra para crominancia ($Cb$, $Cr$). La matriz de luminancia tiene valores más pequeños en la esquina superior izquierda y valores progresivamente mayores hacia la esquina inferior derecha. Esto significa que los coeficientes de baja frecuencia (donde la energía es mayor) se cuantifican con mayor precisión, mientras que los de alta frecuencia (donde la energía suele ser pequeña) se cuantifican con menos precisión. El resultado es que las altas frecuencias se descartan primero al reducir la calidad.

Un ejemplo de matriz de cuantificación para luminancia es:

$$Q_Y = \begin{pmatrix}
16 & 11 & 10 & 16 & 24 & 40 & 51 & 61 \\
12 & 12 & 14 & 19 & 26 & 58 & 60 & 55 \\
14 & 13 & 16 & 24 & 40 & 57 & 69 & 56 \\
14 & 17 & 22 & 29 & 51 & 87 & 80 & 62 \\
18 & 22 & 37 & 56 & 68 & 109 & 103 & 77 \\
24 & 35 & 55 & 64 & 81 & 104 & 113 & 92 \\
49 & 64 & 78 & 87 & 103 & 121 & 120 & 101 \\
72 & 92 & 95 & 98 & 112 & 100 & 103 & 99
\end{pmatrix}$$

Observa que los valores aumentan desde la esquina superior izquierda (16) hasta la esquina inferior derecha (hasta 121). Esta estructura es la que permite comprimir más las altas frecuencias que las bajas.

La matriz de crominancia sigue un principio similar pero con valores más grandes en general, lo que permite comprimir más las componentes de color.

Además, el nivel de calidad puede modelarse escalando la matriz de cuantificación. Cuanto mayor es el escalado, más ceros aparecen y más fácil resulta comprimir el bloque, pero también mayor es la pérdida visual. Pero en el estandar JPEG el nivel de calidad no se aplica directamente como un factor lineal sobre la matriz de cuantificación, sino que se utiliza una función no lineal para ajustar su escala.

En particular, se emplea una transformación del parámetro de calidad $Q \in [1, 100]$ para obtener un factor de escala:

- Para calidades bajas ($Q \leq 50$), el escalado crece rápidamente:
  $$
  \text{scale} = \frac{50}{Q}
  $$

- Para calidades altas ($Q > 50$), el escalado decrece suavemente:
  $$
  \text{scale} = 2 - \frac{Q}{50}
  $$

Este factor se multiplica por la matriz de cuantificación base y posteriormente se redondea y se limita a valores mínimos de 1.

De este modo:
- Valores de calidad bajos generan matrices con valores grandes → más cuantización → más compresión y más pérdida.
- Valores de calidad altos generan matrices más pequeñas → menos cuantización → mejor calidad visual.

Esta formulación permite que el control de calidad sea más perceptualmente uniforme, ya que el ojo humano es más sensible a cambios en altas calidades que en bajas.

In [ ]:
def jpeg_quantization_matrices() -> tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
    QY = np.array([[16, 11, 10, 16, 24, 40, 51, 61],
                   [12, 12, 14, 19, 26, 58, 60, 55],
                   [14, 13, 16, 24, 40, 57, 69, 56],
                   [14, 17, 22, 29, 51, 87, 80, 62],
                   [18, 22, 37, 56, 68, 109, 103, 77],
                   [24, 35, 55, 64, 81, 104, 113, 92],
                   [49, 64, 78, 87, 103, 121, 120, 101],
                   [72, 92, 95, 98, 112, 100, 103, 99]], dtype=np.float64)
    QC = np.array([[17, 18, 24, 47, 99, 99, 99, 99],
                   [18, 21, 26, 66, 99, 99, 99, 99],
                   [24, 26, 56, 99, 99, 99, 99, 99],
                   [47, 66, 99, 99, 99, 99, 99, 99],
                   [99, 99, 99, 99, 99, 99, 99, 99],
                   [99, 99, 99, 99, 99, 99, 99, 99],
                   [99, 99, 99, 99, 99, 99, 99, 99],
                   [99, 99, 99, 99, 99, 99, 99, 99]], dtype=np.float64)
    return QY, QC

def scale_quantization_matrix(Q: npt.NDArray[np.float64], quality: int) -> npt.NDArray[np.float64]:
    if not 1 <= quality <= 100:
        raise ValueError('quality debe estar entre 1 y 100')
    if quality <= 50:
        scale = 50 / quality
    else:
        scale = 2 - quality / 50
    return np.maximum(np.round(Q * scale), 1)

QY, QC = jpeg_quantization_matrices()
print('✓ Matrices de cuantificación cargadas')

### Bloque 3 · Cuantificación de un bloque y análisis visual

**Idea clave:** al cuantificar no se destruyen todos los coeficientes por igual. JPEG penaliza más las altas frecuencias porque suelen tener menos impacto visual. Observar un solo bloque ayuda a entender qué información desaparece antes de pasar a la imagen completa.

**Parámetros:**
- Calidades sugeridas: 90, 50 y 20
- Matriz base: `QY`

**Tareas:**
1. Implementa `quantize_block()` y `unquantize_block()`.
2. Reconstruye el bloque para tres valores de calidad.
3. Cuenta cuántos coeficientes DCT quedan distintos de cero tras cuantificar.
4. Relaciona el número de ceros con la apariencia visual del bloque reconstruido.

In [ ]:
# === IMPLEMENTACIÓN ===
def quantize_block(block_dct: npt.NDArray[np.float64], Q: npt.NDArray[np.float64]) -> npt.NDArray[np.int64]:
                                                                            
    # TODO: divide por Q y redondea al entero más próximo


qualities = [90, 50, 20]
nonzero_counts = []
quantized_blocks = []

for quality in qualities:
    Qq = scale_quantization_matrix(QY, quality)
    block_q = quantize_block(block_dct, Qq)
    quantized_blocks.append(block_q)
    nonzero_counts.append(np.count_nonzero(block_q))

fig, axes = plt.subplots(1, len(qualities) + 1, figsize=(14, 3))
axes[0].imshow(np.log1p(np.abs(block_dct)), cmap='inferno')
axes[0].set_title('DCT sin cuantizar')
axes[0].axis('off')

# DCT cuantizadas
for i, quality in enumerate(qualities):
    axes[i + 1].imshow(np.log1p(np.abs(quantized_blocks[i])), cmap='inferno')
    axes[i + 1].set_title(f'Q={quality}\nno nulos={nonzero_counts[i]}')
    axes[i + 1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# === VALIDACIÓN ===
assert all(block.shape == (8, 8) for block in quantized_blocks)
assert all(0 <= c <= 64 for c in nonzero_counts)
print('✓ Cuantificación de bloque verificada')

### Explicación (OBLIGATORIA)

1. ¿Qué relación observas entre la calidad y el número de coeficientes no nulos?

*Tu respuesta:*
```
[Escribe aquí]
```

2. ¿Por qué la cuantificación produce pérdida aunque la DCT y la IDCT sean reversibles?

*Tu respuesta:*
```
[Escribe aquí]
```

3. Viendo la DCT cuantizada, ¿qué tipo de información se pierde primero al reducir la calidad?
   
*Tu respuesta:*

```
[Escribe aquí]
```

---

## 3b. Cuantificación de la imagen completa

En un codificador real, la cuantificación se aplica a todos los bloques de la imagen. Al visualizar la imagen de coeficientes cuantificados se aprecia cómo aparecen grandes zonas negras cuando muchos coeficientes pasan a cero. Esta representación es muy útil para entender por qué la compresión posterior puede ser tan eficaz.

### Bloque 3c · Cuantificación sobre la imagen completa

**Tareas:**
1. Cuantifica la DCT de la imagen completa para varias calidades.
2. Visualiza la DCT original y las versiones cuantificadas.
3. Describe cómo cambia el número de coeficientes anulados.

In [ ]:
def quantize_image(image_dct: npt.NDArray[np.float64], Q: npt.NDArray[np.float64]) -> npt.NDArray[np.int64]:
    h, w = image_dct.shape
    out = np.zeros((h, w), dtype=np.int64)
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            out[i:i+8, j:j+8] = quantize_block(image_dct[i:i+8, j:j+8], Q)
    return out


def unquantize_image(image_q: npt.NDArray[np.int64], Q: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    h, w = image_q.shape
    out = np.zeros((h, w), dtype=np.float64)
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            out[i:i+8, j:j+8] = unquantize_block(image_q[i:i+8, j:j+8], Q)
    return out


def remain_info_from_quantized(image_q: npt.NDArray[np.int64]) -> float:
    return 100 * np.count_nonzero(image_q) / image_q.size


def simple_jpeg_luma(image_y: npt.NDArray[np.float64], quality: int):
    image_y = crop_to_multiple_of_8(image_y.astype(np.float64))
    centered = image_y - 128.0
    dct_img = dct_of_image(centered)
    Qq = scale_quantization_matrix(QY, quality)
    q_img = quantize_image(dct_img, Qq)
    uq_img = unquantize_image(q_img, Qq)
    rec = np.clip(idct_of_image(uq_img) + 128.0, 0, 255)
    return rec, q_img, dct_img

In [ ]:
# === IMPLEMENTACIÓN ===
#TODO: cuantifica toda la imagen con diferentes calidades y visualiza el resultado

plt.figure(figsize=(14, 14))
plt.subplot(2, 2, 1)
plot_dct(image_y_dct, title='Valores DCT originales')
plt.subplot(2, 2, 2)
plot_dct(quantized_y_90, title='Valores DCT cuantificados (Q=90)')
plt.subplot(2, 2, 3)
plot_dct(quantized_y_50, title='Valores DCT cuantificados (Q=50)')
plt.subplot(2, 2, 4)
plot_dct(quantized_y_20, title='Valores DCT cuantificados (Q=20)')
plt.tight_layout()
plt.show()


In [ ]:
# === VALIDACIÓN ===
assert quantized_y_90.shape == image_y_dct.shape
assert quantized_y_50.shape == image_y_dct.shape
assert quantized_y_20.shape == image_y_dct.shape
print('✓ Cuantificación de la imagen completa calculada')


### Explicación (OBLIGATORIA)

1. ¿Qué observas en la representación de la DCT cuantizada a medida que reduces la calidad?

*Tu respuesta:*
```
[Escribe aquí]
```

2. Comparando la DCT con la imagen, ¿qué zonas de la imagen se ven más afectadas por la cuantificación?

*Tu respuesta:*
```
[Escribe aquí]
```

1. Al reconstruir la imagen a partir de la DCT cuantizada, ¿pueden verse afectados también los detalles de baja frecuencia? Explica tu respuesta.
   
*Tu respuesta:*

```
[Escribe aquí]
```

### Bloque 3b · ¿Cuánto aporta realmente el coeficiente DC?

El coeficiente DC en la DCT representa el valor medio de intensidad de un bloque de imagen, es decir, su nivel global de luminancia. En un bloque de $8 \times 8$, este coeficiente se sitúa en la posición (0,0) y suele ser el de mayor magnitud, ya que concentra gran parte de la energía de la señal. Esto se debe a que las imágenes naturales presentan variaciones suaves y continuas, por lo que la información de baja frecuencia domina sobre los detalles finos. En consecuencia, el componente DC determina en gran medida si un bloque se percibe como claro u oscuro y establece la base sobre la que se construye el resto de la información visual.

Desde el punto de vista perceptual, el DC es especialmente importante porque, incluso cuando se eliminan todos los demás coeficientes, la imagen reconstruida conserva una estructura global reconocible, aunque muy simplificada y con aspecto de mosaico. Esto muestra que gran parte de la información relevante para el ojo humano está contenida en este valor medio. Por el contrario, si se elimina el DC y se mantienen solo los coeficientes de alta frecuencia, la imagen pierde su referencia de luminancia y resulta mucho más difícil de interpretar.

Además, el coeficiente DC tiene un papel clave en la compresión JPEG, ya que varía lentamente entre bloques adyacentes. Esto permite codificarlo de forma muy eficiente mediante diferencias entre bloques consecutivos, reduciendo significativamente el número de bits necesarios. En conjunto, el DC no solo concentra gran parte de la información energética de la imagen, sino que también es fundamental tanto para su percepción como para su compresión eficiente.


**Tareas:**
1. Haz la DCT de cada bloque de la imagen, y a la hora de cuantificar, en lugar de usar la forma estandar, aplica tres máscaras distintas a los coeficientes DCT de cada bloque para quedarte con:
   - solo el componente DC,
   - el componenteDC + los dos coeficientes vecinos de mas baja frecuencia (posiciones (0,1) y (1,0)),
   - todos menos esos tres coeficientes.
2. Reconstruye la imagen de ejemplo con cada una de las tres máscaras.
3. Compara visualmente los resultados y calcula MSE, SSIM y porcentaje de coeficientes conservados.

In [ ]:
# === IMPLEMENTACIÓN ===
# Esta función reconstruye la imagen a partir de la DCT aplicando una máscara a los coeficientes (en lugar de cuantificar)
def reconstruct_with_frequency_mask(image_y_in: npt.NDArray[np.float64], mask: npt.NDArray[np.float64]):
    image_y_in = crop_to_multiple_of_8(image_y_in.astype(np.float64))
    centered = image_y_in - 128.0
    dct_img = dct_of_image(centered)
    masked = apply_blockwise(dct_img, lambda block: block * mask)
    rec = np.clip(idct_of_image(masked) + 128.0, 0, 255)
    kept_ratio = 100 * np.count_nonzero(mask) / mask.size
    return rec, kept_ratio

#TODO: construye las tres máscaras y reconstruye la imagen con cada una de ellas


image_dc_only, kept_dc = reconstruct_with_frequency_mask(image_y, mask_only_dc)
image_low3, kept_low3 = reconstruct_with_frequency_mask(image_y, mask_low3)
image_high61, kept_high61 = reconstruct_with_frequency_mask(image_y, mask_high61)

metrics_dc = (mse_metric(crop_to_multiple_of_8(image_y), image_dc_only), ssim_metric(crop_to_multiple_of_8(image_y), image_dc_only))
metrics_low3 = (mse_metric(crop_to_multiple_of_8(image_y), image_low3), ssim_metric(crop_to_multiple_of_8(image_y), image_low3))
metrics_high61 = (mse_metric(crop_to_multiple_of_8(image_y), image_high61), ssim_metric(crop_to_multiple_of_8(image_y), image_high61))

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes[0, 0].imshow(crop_to_multiple_of_8(image_y), cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original (Y)')
axes[0, 1].imshow(image_dc_only, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title(f'Solo DC\nSSIM={metrics_dc[1]:.3f}')
axes[1, 0].imshow(image_low3, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title(f'DC + 2 bajas\nSSIM={metrics_low3[1]:.3f}')
axes[1, 1].imshow(image_high61, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title(f'Solo altas (61 coefs)\nSSIM={metrics_high61[1]:.3f}')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Solo DC -> coeficientes conservados: {kept_dc:.4f}% | MSE={metrics_dc[0]:.3f} | SSIM={metrics_dc[1]:.3f}')
print(f'DC + 2 bajas -> coeficientes conservados: {kept_low3:.4f}% | MSE={metrics_low3[0]:.3f} | SSIM={metrics_low3[1]:.3f}')
print(f'Solo altas (61 coef.) -> coeficientes conservados: {kept_high61:.4f}% | MSE={metrics_high61[0]:.3f} | SSIM={metrics_high61[1]:.3f}')


In [ ]:
# === VALIDACIÓN ===
assert image_dc_only.shape == crop_to_multiple_of_8(image_y).shape
assert image_low3.shape == image_dc_only.shape == image_high61.shape
assert kept_dc < kept_low3 < kept_high61
print('✓ Experimento de importancia del coeficiente DC ejecutado')


### Explicación (OBLIGATORIA)

1. ¿Cómo explicas que conservar solo unas pocas bajas frecuencias pueda dar una imagen perceptualmente razonable?

*Tu respuesta:*
```
[Escribe aquí]
```

2. ¿Ves alguna aparente contradicción entre porcentaje de coeficientes conservados y el valor de MSE? ¿Cuál es la explicación de esa aparente contradicción?

*Tu respuesta:*
```
[Escribe aquí]
```

### Bloque 3d · Deshacer la cuantificación de un bloque

Para reconstruir la imagen hay que multiplicar de nuevo cada bloquepor la matriz de cuantificación utilizada. Este paso no recupera exactamente los coeficientes originales cuando ya se ha producido redondeo, pero sí devuelve una aproximación de la DCT previa a la cuantificación. Tras esta operación, la IDCT devuelve una imagen que se acerca a la original pero con pérdida de detalles (que será mayor cuanto más agresiva sea la cuantificación).

**Tareas:**
1. Cuantifica el bloque de ejemplo con una calidad intermedia $Q = 50$.
2. Deshaz la cuantificación multiplicando por la matriz escalada.
3. Compara la DCT original, la cuantificada y la recuperada.

In [ ]:
# === IMPLEMENTACIÓN ===

def unquantize_block(block_q: npt.NDArray[np.int64], Q: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    # TODO: reconstruye una aproximación de la DCT original


#TODO: cuantifica el bloque de ejemplo con una calidad intermedia, deshaz la cuantificación y compara la DCT original, la cuantificada y la recuperada


plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plot_dct(block_dct, 'DCT original')
plt.subplot(1, 3, 2)
plot_dct(block_q50, 'DCT cuantificada')
plt.subplot(1, 3, 3)
plot_dct(block_uq50, 'DCT recuperada')
plt.tight_layout()
plt.show()


In [ ]:
# === VALIDACIÓN ===
assert block_q50.shape == (8, 8)
assert block_uq50.shape == (8, 8)
print('✓ Descuantificación de bloque realizada')


### Explicación (OBLIGATORIA)

1. Al deshacer la cuantificación, ¿todos los valores no nulos se recuperaron exactamente igual que en la DCT original? ¿Por qué?

*Tu respuesta:*
```
[Escribe aquí]
```

2. ¿Cuáles son los coeficientes de la DCT que no se pudieron recuperar en absoluto? ¿A qué es debido esto?

*Tu respuesta:*
```
[Escribe aquí]
```

### Bloque 3e · Pipeline JPEG simplificado sobre la imagen completa

Con las funciones implementadas hasta ahora ya es posible ejecutar el pipeline principal: centrar la imagen, aplicar la DCT a todos los bloques, cuantificar, deshacer la cuantificación, aplicar la IDCT y reconstruir la imagen final. Este proceso no incluye todavía zig-zag, RLE ni Huffman (que son las fases que permiten obtener el stream final de bits), pero sí captura la esencia perceptual del JPEG.

**Tareas:**
1. Ejecuta el pipeline completo sobre la componente Y.
2. Reconstruye la imagen para varias calidades, 90, 50 y 10.
3. Calcula MSE y SSIM para cada calidad.

In [ ]:
# === IMPLEMENTACIÓN ===
qualities_pipeline = [90, 50, 10]
pipeline_results = []

orig = crop_to_multiple_of_8(image_y)

fig, axes = plt.subplots(1, len(qualities_pipeline) + 1, figsize=(16, 4))
axes[0].imshow(orig, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original (Y)')
axes[0].axis('off')

for idx, q in enumerate(qualities_pipeline, start=1):
    rec_img, q_img, _ = simple_jpeg_luma(image_y, q)
    
    # Métricas
    mse_val = mse_metric(orig, rec_img)
    ssim_val = ssim_metric(orig, rec_img)
    pipeline_results.append((q, rec_img, mse_val, ssim_val))

    axes[idx].imshow(rec_img, cmap='gray', vmin=0, vmax=255)
    axes[idx].set_title(f'Q={q}\nMSE={mse_val:.1f}\nSSIM={ssim_val:.3f}')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

for q, _, mse_val, ssim_val in pipeline_results:
    print(f'Calidad {q}: MSE={mse_val:.3f} | SSIM={ssim_val:.4f}')


In [ ]:
# === VALIDACIÓN ===
assert len(pipeline_results) == 3
print('✓ Pipeline JPEG simplificado ejecutado')


### Explicación (OBLIGATORIA)

1. ¿Son coherentes los resultados obtenidos de MSE y SSIM con el nivel de calidad aplicado?  ¿A qué se debe?

*Tu respuesta:*
```
[Escribe aquí]
```

---

## 5. Implementación final · Codificación entrópica y bitstream

En las secciones anteriores ya has implementado la parte con pérdida del codificador: conversión de color, DCT, cuantificación y reconstrucción mediante IDCT. En esta última parte nos centraremos en la etapa que viene después de la cuantificación: la codificación entrópica.

En JPEG baseline, los coeficientes cuantificados de cada bloque no se guardan directamente como una matriz de $8\times8$. Primero se recorren en zig-zag, después se separa el coeficiente DC de los coeficientes AC, y finalmente se codifican con una combinación de DPCM, RLE y Huffman. Esta etapa no introduce pérdida adicional: solo intenta representar de forma más compacta los enteros que ya han salido de la cuantificación.

Para que la práctica no se convierta en copiar tablas largas, usaremos las utilidades ya incluidas en `jpeg_lib.py`:

- `block_to_zigzag()` y `zigzag_to_block()` para el recorrido zig-zag,
- `huffman_decimal_to_binary()` y `huffman_binary_to_decimal()` para los bits de amplitud,
- `huffman_codes()` para obtener las tablas Huffman de luminancia y crominancia,
- `jpeg_coded_image_size()` para estimar los bits por píxel del bitstream.

De este modo, el trabajo de esta sección queda centrado en entender y programar la lógica JPEG: diferencia DC, RLE de AC, símbolos especiales *EOB* y *ZRL*, y codificación bloque a bloque.

### Secuencia conceptual

Para cada bloque cuantificado:

1. Se aplica el recorrido zig-zag para convertir el bloque $8\times8$ en un vector de 64 coeficientes. Este recorrido ordena los coeficientes desde bajas a altas frecuencias para que los ceros queden agrupados al final del vector.

<img src="../helpers/jpeg_dct_zig_zag.png" alt="Recorrido en zig-zag." style="width: 35%"/>

2. El primer valor del vector es el coeficiente DC. No se codifica directamente, sino como diferencia respecto al DC del bloque anterior.
3. Los 63 valores restantes son coeficientes AC. Como tras la cuantificación muchos de ellos son cero, se codifican mediante RLE.
4. Cada símbolo estructural se codifica con Huffman:
   - DC: categoría de la diferencia,
   - AC: par `(run, categoría)`, donde `run` es el número de ceros anteriores al coeficiente no nulo.
5. Después de cada símbolo Huffman se añaden los bits de amplitud necesarios para recuperar el valor concreto.

Los símbolos especiales son:

- *EOB* (`End Of Block`): indica que todos los coeficientes AC restantes son cero.
- *ZRL* (`Zero Run Length`): representa una racha de 16 ceros consecutivos.

En esta versión simplificada no construiremos un fichero JPEG real con cabeceras y marcadores. El resultado será una lista de cadenas de bits por bloque, suficiente para estudiar la compresión entrópica y estimar la tasa final.


### Bloque 5a · Ejemplo manual guiado de un bloque

Antes de programar la codificación entrópica, conviene recorrer un ejemplo pequeño. Supón que, tras cuantificar y recorrer en zig-zag un bloque, obtienes el siguiente vector simplificado:

```text
[15, 0, 0, -3, 0, 2, 0, 0, 0, 0, 0, 1, 0, ..., 0]
```

El primer valor, `15`, es el coeficiente DC del bloque. Si el DC del bloque anterior fuese `12`, JPEG no codificaría directamente `15`, sino la diferencia:

$$
DC_{diff} = 15 - 12 = 3.
$$

Esa diferencia se transforma en dos partes: su categoría y sus bits de amplitud. La categoría indica cuántos bits hacen falta para representar la magnitud del valor, mientras que los bits de amplitud indican el valor concreto.

La parte AC sería:

```text
[0, 0, -3, 0, 2, 0, 0, 0, 0, 0, 1, 0, ..., 0]
```

La codificación RLE no guarda todos los ceros uno a uno. En su lugar, cada coeficiente no nulo se describe mediante cuántos ceros aparecen antes de él. En este ejemplo aparecen los símbolos conceptuales:

```text
(2, -3), (1, 2), (5, 1), EOB
```

En la codificación real, Huffman no recibe directamente `(2, -3)`. Primero se convierte el valor `-3` en su categoría, y Huffman codifica el par `(run, categoría)`. Después se añaden los bits de amplitud de `-3`.



Para el siguiente ejemplo de secuencia de coeficientes cuantificados, traza a mano el proceso completo de codificación:

```text
[20, 0, 0, 0, -5, 0, 0, 3, 0, 0, 0, 0, 1, ..., 0]
```

**Tareas:**
1. Calcula la diferencia DC del ejemplo anterior si el bloque previo tiene DC = 12.
2. Escribe la secuencia RLE conceptual de la parte AC.
3. Indica dónde aparece el símbolo `EOB`.

### Explicación (OBLIGATORIA)

1. Escribe el resultado del trazado manual del bloque de ejemplo.

*Tu respuesta:*
```
[Escribe aquí]
```


### Bloque 5b · Codificación entrópica de bloques cuantificados

En esta versión del taller no tienes que implementar desde cero ni el zig-zag, ni las tablas Huffman, ni la conversión entre enteros y bits de amplitud. Esas piezas ya están disponibles en `jpeg_lib.py`.

Algunas de las funciones disponibles en `jpeg_lib.py` son:

* `huffman_decimal_to_binary(d: int) -> str:`

Convierten de representación decimal a binaria. La representación binaria se realiza mediante cadenas de texto. 

* `huffman_codes(type: str) -> [tuple[str], tuple[tuple[str]]]`

Devuelve los códigos Huffman para los coeficientes DC y AC. Si recibe como parámetro 'L' devuelve los códigos para luminancia, y si recibe 'C' los de crominancia.

**Tareas:**
1. Implementa `encode_dc_coefficient()` aplicando DPCM al DC y la tabla Huffman adecuada.
2. Implementa `encode_ac_coefficients()` con RLE, gestionando correctamente `EOB` y `ZRL`.
3. Completa `jpeg_huffman_encode_block()` combinando DC y AC.
4. Usa `jpeg_encoder()` para generar el bitstream de todos los bloques de una imagen en luminancia.

**Pistas:**
- El componente de luminancia usa `component_type='L'` como parámetro.
- Para el DC se usa la tabla `huff_dc[category]`.
- Para AC, el símbolo `(run, category)` se consulta como `huff_ac[run][category - 1]`.
- `EOB` se obtiene como `huff_ac[0][0]`.
- `ZRL` se obtiene como `huff_ac[15][0]`.
- La función `jpeg_encoder()` de este ejercicio genera un bitstream con los datos de luminancia, no un fichero `.jpg` completo real.


In [ ]:
# === IMPLEMENTACIÓN FINAL  ===
# En esta sección utilizaremos las utilidades de jpeg_lib.py 

from jpeg_lib import (
    block_to_zigzag as lib_block_to_zigzag,
    huffman_decimal_to_binary,
    huffman_codes,
    jpeg_coded_image_size,
)

def coefficient_category(value: int) -> int:
    """
    Devuelve la categoría JPEG de un coeficiente entero.

    Categoría = número de bits necesarios para representar la magnitud.
    Ejemplos:
        0  -> categoría 0
        1  -> categoría 1
        -1 -> categoría 1
        2  -> categoría 2
        -3 -> categoría 2
    """
    value = int(value)

    if value == 0:
        return 0

    return len(huffman_decimal_to_binary(value))


def coefficient_amplitude_bits(value: int) -> str:
    """
    Devuelve los bits de amplitud JPEG asociados a un coeficiente.

    Para valores positivos coincide con su binario.
    Para valores negativos se usa la representación JPEG basada en
    complementar los bits de la magnitud.
    """
    value = int(value)

    if value == 0:
        return ""

    return huffman_decimal_to_binary(value)

#Codifica el coeficiente DC de un bloque.
def encode_dc_coefficient(
    current_dc: int,
    previous_dc: int,
    component_type: str = "L"
) -> list[str]:
    # TODO: calcula la diferencia DC, su categoría y bits de amplitud


    huff_dc, _ = huffman_codes(component_type)

    if category == 0:
        return [huff_dc[0]]

    return [huff_dc[category], amplitude]


# Codifica los 63 coeficientes AC de un bloque usando RLE + Huffman.
def encode_ac_coefficients(
    ac_values: npt.NDArray[np.int64],
    component_type: str = "L"
) -> list[str]:
    _, huff_ac = huffman_codes(component_type)

    bitstream = []
    zero_run = 0

    # TODO Implementa lo siguiente:
    # - Cuenta cuántos ceros consecutivos aparecen antes de cada valor no nulo.
    # - Si hay 16 ceros consecutivos, emite ZRL.
    # - Para cada valor no nulo, emite: Huffman(run, categoría) + bits de amplitud.
    # - Si al final quedan ceros, emite EOB.
    for value in ac_values:
        value = int(value)

        if value == 0:
            zero_run += 1
            continue

        # TODO: Si hay más de 15 ceros seguidos, JPEG emite uno o más ZRL.

        #TODO: calcula la categoría y los bits de amplitud del valor no nulo.


        # En las tablas AC, la categoría 1 está en la columna 0.
        bitstream.append(huff_ac[zero_run][category - 1])
        bitstream.append(amplitude)

        zero_run = 0

    # TODO:Si el bloque termina con ceros, se emite EOB.


    return bitstream

"""
Codifica entrópicamente un bloque 8x8 ya cuantificado.

Entrada:
    a_block: bloque cuantificado 8x8.
    component_type: 'L' para luminancia, 'C' para crominancia.
    previous_dc: DC cuantificado del bloque anterior.

Salida:
    lista de cadenas de bits.
"""
def jpeg_huffman_encode_block(
    a_block: npt.NDArray[np.int64],
    component_type: str = "L",
    previous_dc: int = 0
) -> list[str]:

    zz = lib_block_to_zigzag(a_block.astype(np.int64))

    current_dc = int(zz[0])
    ac_values = zz[1:]

    # TODO: codifica el bloque usando las funciones de codificación DC y AC.


    return dc_bits + ac_bits

"""
Codifica todos los bloques cuantificados de una imagen de un canal.

Mantiene el DC anterior para aplicar DPCM bloque a bloque.
"""
def entropy_encode_quantized_image(
    image_q: npt.NDArray[np.int64],
    component_type: str = "L"
) -> list[list[str]]:
    h, w = image_q.shape
    bitstream = []
    previous_dc = 0

    for i in range(0, h, 8):
        for j in range(0, w, 8):
            block_q = image_q[i:i+8, j:j+8]
            block_bits = jpeg_huffman_encode_block(
                block_q,
                component_type=component_type,
                previous_dc=previous_dc
            )
            bitstream.append(block_bits)
            previous_dc = int(block_q[0, 0])

    return bitstream


def jpeg_encoder(
    image: npt.NDArray[np.float64],
    quality: int = 50,
    subsampling: str = "4:4:4"
) -> list[list[str]]:
    """
    Codificador JPEG completo para una imagen de un canal (luminancia).

    Pasos:
    1. Recorta la imagen a múltiplos de 8.
    2. Centra los píxeles restando 128.
    3. Calcula la DCT por bloques.
    4. Cuantifica.
    5. Codifica entrópicamente los bloques cuantificados.

    Nota:
    - Esta versión trabaja con un único canal.
    - El parámetro subsampling se conserva para mantener la interfaz,
      pero no se usa en imágenes de luminancia.
    """
    if image.ndim != 2:
        raise NotImplementedError(
            "Esta versión codifica solo un canal. "
            "Para color, codifica Y, Cb y Cr por separado."
        )

    image_y = crop_to_multiple_of_8(image.astype(np.float64))
    centered = image_y - 128.0

    image_dct = dct_of_image(centered)

    Qq = scale_quantization_matrix(QY, quality)
    image_q = quantize_image(image_dct, Qq)

    return entropy_encode_quantized_image(image_q, component_type="L")


def estimated_bpp(
    bitstream: list[list[str]],
    image_shape: tuple[int, int],
    channels: int = 1
) -> float:
    """
    Estima los bits por píxel incluyendo una aproximación del coste de tablas.
    """
    return jpeg_coded_image_size(bitstream, image_shape, channels=channels)



### Checkpoint del profesor

- [ ] El estudiante entiende que el zig-zag y las tablas Huffman se reutilizan desde `jpeg_lib.py`.
- [ ] El estudiante calcula correctamente la categoría y los bits de amplitud.
- [ ] El estudiante codifica el DC como diferencia respecto al bloque anterior.
- [ ] El estudiante implementa RLE de AC con los casos `EOB` y `ZRL`.
- [ ] El estudiante genera un bitstream no vacío para todos los bloques de una imagen.
- [ ] El estudiante estima bits por píxel y relaciona esa tasa con calidad y cuantificación.


---

## 6. Validación final y análisis experimental

En esta versión simplificada, la reconstrucción de la imagen se valida con el pipeline de DCT, cuantificación, descuantificación e IDCT que ya has usado antes (`simple_jpeg_luma`). La codificación entrópica se evalúa por separado midiendo el tamaño estimado del bitstream generado por `jpeg_encoder()`.

Esta separación es útil desde el punto de vista docente:

- La calidad visual depende de la cuantificación.
- La tasa de compresión depende de cuántos coeficientes sobreviven y de cómo se codifican con RLE + Huffman.
- La etapa Huffman no cambia la imagen reconstruida, pero sí cambia el número de bits necesarios para representarla.

Por tanto, en esta fase compararemos simultáneamente:

1. la imagen reconstruida,
2. MSE y SSIM,
3. bits por píxel estimados,
4. factor de compresión aproximado.


### Bloque 6a · Comprobaciones básicas de funcionamiento

**Tareas:**
1. Ejecuta `jpeg_encoder()` sobre una imagen de luminancia recortada a múltiplos de 8.
2. Comprueba que el bitstream contiene un elemento por cada bloque de $8\times8$.
3. Comprueba que el bitstream no está vacío.
4. Reconstruye la imagen con `simple_jpeg_luma()` para medir MSE y SSIM.
5. Calcula los bits por píxel estimados con `estimated_bpp()`.


In [ ]:
# === COMPROBACIONES BÁSICAS ===

test_image = crop_to_multiple_of_8(image_y.astype(np.float64))
test_quality = 90

bitstream = jpeg_encoder(test_image, quality=test_quality)
reconstructed, q_img, _ = simple_jpeg_luma(test_image, test_quality)

num_blocks_expected = (test_image.shape[0] // 8) * (test_image.shape[1] // 8)
bpp = estimated_bpp(bitstream, test_image.shape, channels=1)

print("Dimensiones:", test_image.shape)
print("Bloques esperados:", num_blocks_expected)
print("Bloques codificados:", len(bitstream))
print("Bitstream no vacío:", len(bitstream) > 0 and sum(len(b) for b in bitstream) > 0)
print(f"MSE:  {mse_metric(test_image, reconstructed):.3f}")
print(f"SSIM: {ssim_metric(test_image, reconstructed):.4f}")
print(f"Bits/pixel estimados: {bpp:.3f}")
print(f"Factor de compresión aproximado: {8 / bpp:.2f}x")


### Bloque 6b · Tabla de resultados experimentales

Realiza pruebas con varias calidades. En este caso, como trabajamos solo con luminancia, deja el subsampling como `4:4:4`. Si amplías el codificador a color, puedes repetir la tabla con `4:2:2` y `4:2:0`.

Para cada caso registra:

- MSE,
- SSIM,
- bits por píxel estimados,
- factor de compresión aproximado,
- tiempo de codificación entrópica,
- una observación cualitativa breve.

La tabla debe incluir resultados para:

- dos imágenes distintas en escala de grises,
- tres niveles de calidad: 90, 50 y 20.

| Imagen | Calidad | MSE | SSIM | bits/pixel | Factor compresión | t_encode (s) | Observaciones |
|---|---:|---:|---:|---:|---:|---:|---|
| imagen 1 | 90 |  |  |  |  |  |  |
| imagen 1 | 50 |  |  |  |  |  |  |
| imagen 1 | 20 |  |  |  |  |  |  |
| imagen 2 | 90 |  |  |  |  |  |  |
| imagen 2 | 50 |  |  |  |  |  |  |
| imagen 2 | 20 |  |  |  |  |  |  |

Para una imagen de un canal, el factor de compresión aproximado puede estimarse como:

$$
\text{factor de compresión} \approx \frac{8}{\text{bits/pixel}}.
$$

La estimación no corresponde exactamente al tamaño de un fichero `.jpg` real, pero permite comparar configuraciones de forma coherente dentro del taller.


In [ ]:
# === TABLA EXPERIMENTAL ===

qualities = [90, 50, 20]
results_table = []

#TODO: selecciona un par de imágenes de prueba, codifícalas con diferentes calidades y rellena la tabla con las métricas correspondientes


for image_name, test_image in test_images.items():
    for quality in qualities:
        t0 = time.perf_counter()
        bitstream = jpeg_encoder(test_image, quality=quality)
        t1 = time.perf_counter()

        reconstructed, q_img, _ = simple_jpeg_luma(test_image, quality)

        bpp = estimated_bpp(bitstream, test_image.shape, channels=1)
        compression_ratio = 8 / bpp

        results_table.append({
            "image": image_name,
            "quality": quality,
            "mse": mse_metric(test_image, reconstructed),
            "ssim": ssim_metric(test_image, reconstructed),
            "bpp": bpp,
            "compression_ratio": compression_ratio,
            "encode_time": t1 - t0,
            "nonzero_coefficients": remain_info_from_quantized(q_img),
        })

for row in results_table:
    print(
        f"{row['image']:10s} | "
        f"Q={row['quality']:2d} | "
        f"MSE={row['mse']:8.3f} | "
        f"SSIM={row['ssim']:.4f} | "
        f"bpp={row['bpp']:.3f} | "
        f"CR={row['compression_ratio']:.2f}x | "
        f"no nulos={row['nonzero_coefficients']:.2f}% | "
        f"t={row['encode_time']:.4f}s"
    )


### Explicación (OBLIGATORIA)

1. Resume los resultados más importantes de tu tabla y explica qué tendencias generales observas.

*Tu respuesta:*
```
[Escribe aquí]
```

2. Indica qué calidad elegirías si tu objetivo fuese maximizar compresión y cuál elegirías si tu objetivo fuese maximizar calidad perceptual.

*Tu respuesta:*
```
[Escribe aquí]
```


---

## Checklist final

- [ ] Conversión RGB → YCbCr comprendida.
- [ ] Subsampling cromático comparado visualmente.
- [ ] Selección y análisis de bloques $8\times8$ realizados.
- [ ] DCT e IDCT implementadas y verificadas.
- [ ] Cuantificación y descuantificación analizadas.
- [ ] Importancia del coeficiente DC comprendida.
- [ ] Categoría y bits de amplitud comprendidos.
- [ ] DC codificado de forma diferencial.
- [ ] AC codificados con RLE, `EOB` y `ZRL`.
- [ ] Bitstream generado bloque a bloque.
- [ ] Bits por píxel estimados.
- [ ] Resultados interpretados mediante métricas y observación visual.


---

## Funcionalidad opcional

El codificador implementado en esta práctica es una versión muy simplificada del estándar JPEG. Para acercarlo más a la realidad, puedes implementar las siguientes funcionalidades adicionales:

1. Añade soporte completo para color codificando por separado `Y`, `Cb` y `Cr`.
2. Incorpora subsampling real en crominancia antes de codificar los canales de color.
3. Compara `4:4:4`, `4:2:2` y `4:2:0` en la tabla de resultados.

En caso de estar correctamente implementadas, la nota de entrega de la práctica se incrementará un 10%.
